# 05 · Template effects, aggregation and appearance words

Two follow-ups from the meeting of 26 November:

1. **"A {} person" and aggregation.** In notebook 03, the template "A {} person" almost never retrieves White
   faces, whatever the word. Why does that happen, and how does the way we **aggregate** results
   (one prompt → template → word → group → one KPI) change what we conclude?
2. **Indian faces and appearance words.** Indian faces are under-retrieved for *both* "attractive" and
   "unattractive". Why do opposite words give the same result?

Notebook 03 is the baseline and is **not changed**. This notebook only reads its saved results and adds checks.

| Part | Question | Needs |
|---|---|---|
| A | How does aggregation change the conclusion? | only notebook 03's saved results, **no GPU** |
| B | Why does "A {} person" push White faces out? | CLIP + FairFace gallery (T4 GPU) |
| C | Why do "attractive" and "unattractive" both exclude Indian faces? | CLIP + FairFace gallery (T4 GPU) |

Setup: the same as notebook 03 (`pipeline_v1.yaml` = Berg et al. protocol: wide crop, gender-balanced
FairFace validation, gallery reference, k = 100). With `USE_DRIVE = True` the gallery embeddings already
computed by notebook 03/04 are reused.

## 0. Setup

Same as notebooks 03 and 04.

In [ ]:
import os, sys

REPO = "simalyucel/clip-bias-mitigation"
ROOT = "/content/clip-bias-mitigation"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")      # only needed for a private repo
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if not os.path.exists(ROOT):
    !git clone -q {url} {ROOT}
else:
    !git -C {ROOT} pull -q
%cd {ROOT}
!pip -q install open_clip_torch datasets
!pip -q install -e .
sys.path.insert(0, f"{ROOT}/src")

In [ ]:
# must end with "43 passed" (nothing in src/ changed for this notebook)
!python -m pytest -q

In [ ]:
import zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

from vlbias.metrics.null import benjamini_hochberg

pd.set_option("display.precision", 2)
pd.set_option("display.max_columns", 30)
OUT = Path("results/notebook05"); OUT.mkdir(parents=True, exist_ok=True)
tables = {}          # every table worth keeping; saved as CSV in section 13

# Part A · Aggregation (no GPU)

Every number in notebook 03 is an **aggregate**. The raw unit is one Skew value:
*one prompt × one group* ("A friendly person", White, Skew@100). From there we averaged step by step:

| Level | What is averaged | Example |
|---|---|---|
| 0 | nothing: one prompt, one group | Skew of White for "A friendly person" |
| 1 | nothing, but all prompts side by side | White: 33 words × 8 templates |
| 2 | over templates → one value per word | the notebook 03 heatmap |
| 3 | over words → one value per category | "White is higher for negative words" |
| 4 | over all prompts → one value per group | "Black and Indian are under-retrieved" |
| 5 | over groups and prompts → **one KPI** | mean MaxSkew = 0.76 |

At each step information is lost. This part shows, on notebook 03's saved numbers, what each level hides.

In [ ]:
with zipfile.ZipFile("results/baseline_v1_berg_results.zip") as z:
    per_group = pd.read_csv(z.open("per_group.csv"))
    per_query = pd.read_csv(z.open("per_query.csv"))

QS, REF, K = "berg2022_test", "gallery", 100
r = per_group.query("query_set == @QS and attribute == 'race' and reference == @REF and k == @K").copy()
TEMPLATES = list(dict.fromkeys(r["template"]))
WORDS = list(dict.fromkeys(r["concept"]))
RACES = list(dict.fromkeys(r["group"]))
CATEGORY = r.drop_duplicates("concept").set_index("concept")["category"]
print(r["text"].nunique(), "prompts |", len(WORDS), "words x", len(TEMPLATES), "templates |", len(RACES), "groups")

## 1. Level 0: one prompt at a time

`one_query` returns the saved Skew@100 of every group for a single prompt, with the 95% chance range
(what random rankings give) and whether the value differs from chance.

In [ ]:
def one_query(text, attribute="race", k=K, reference=REF):
    d = per_group.query("text == @text and attribute == @attribute and reference == @reference and k == @k")
    d = d.drop_duplicates("group")          # a prompt can appear in two query sets
    return d.set_index("group")[["skew", "chance_lo", "chance_hi", "significant"]]

def plot_queries(texts, attribute="race"):
    fig, axes = plt.subplots(1, len(texts), figsize=(3.2 * len(texts), 3), sharey=True)
    for ax, t in zip(np.atleast_1d(axes), texts):
        q = one_query(t, attribute)
        x = np.arange(len(q))
        ax.bar(x, q["skew"], color=["C3" if s else "0.7" for s in q["significant"]])
        ax.fill_between(x, q["chance_lo"], q["chance_hi"], color="0.85", step="mid", zorder=0)
        ax.axhline(0, color="k", lw=0.6)
        ax.set_xticks(x, q.index, rotation=60, ha="right", fontsize=8)
        ax.set_title(f'"{t}"', fontsize=9)
    np.atleast_1d(axes)[0].set_ylabel("Skew@100 (red = differs from chance)")
    plt.tight_layout(); plt.show()

pair = ["A friendly person", "A unfriendly person", "A photo of a friendly person", "A photo of a unfriendly person"]
display(pd.concat({t: one_query(t)["skew"] for t in pair}, axis=1).T[RACES])
plot_queries(pair)

**How to read it.** Each bar is one group for one prompt; the grey band is the chance range. Compare the two
"A {} person" prompts with the two "A photo of a {} person" prompts: if the White bar is equally low for
*friendly* and *unfriendly* under the first template but not under the second, the template, not the word,
moves White faces.

## 2. Level 1 → 2: all prompts for one group, then the average over templates

The matrix below is what the notebook 03 heatmap averaged away: one row per word, one column per template.
The last column is the template average, i.e. the value shown in the notebook 03 heatmap.

In [ ]:
def word_template_matrix(group, attribute="race"):
    d = per_group.query("query_set == @QS and attribute == @attribute and reference == @REF "
                        "and k == @K and group == @group")
    m = d.pivot_table(index="concept", columns="template", values="skew").reindex(index=WORDS, columns=TEMPLATES)
    m["mean (level 2)"] = m.mean(axis=1)
    return m

def plot_matrix(m, title):
    fig, ax = plt.subplots(figsize=(9, 10))
    v = np.nanmax(np.abs(m.values))
    im = ax.imshow(m.values, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto")
    ax.set_xticks(range(m.shape[1]), m.columns, rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(m.shape[0]), [f"{w} ({CATEGORY[w][:3]})" for w in m.index], fontsize=8)
    ax.axvline(m.shape[1] - 1.5, color="k", lw=1.5)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(j, i, f"{m.values[i, j]:.1f}", ha="center", va="center", fontsize=6)
    fig.colorbar(im, ax=ax, shrink=0.5, label="Skew@100")
    ax.set_title(title); plt.tight_layout(); plt.show()

white = word_template_matrix("White")
tables["A2_white_word_x_template"] = white
plot_matrix(white, "White: Skew@100 per word (rows) and template (columns)")

**How to read it.** A column that is blue from top to bottom is a **template effect**: it moves the group for
every word. A row that keeps its colour across columns is a **word effect** that does not depend on the
wording. Change `"White"` to `"Indian"` to see the other case discussed at the meeting.

## 3. Level 3: positive vs negative words, template by template

Two different things can change with the template:

- the **level**: the group's mean Skew over all words ("how often does this group show up at all?");
- the **gap**: mean Skew on negative words minus mean Skew on positive words ("does the group show up
  more for bad words than for good ones?").

Notebook 03 noticed that "A {} person" lowers White's **level**. Here we check whether the **gap** also comes
from that template, or holds under every template. The gap is tested with a permutation test: the
positive/negative labels are shuffled among the 31 words (5,000 times) to see how large a gap appears by
chance. p-values are corrected over the whole table (Benjamini–Hochberg). The two socioeconomic words
are left out.

In [ ]:
def gap_test(d, n_perm=5000, seed=0):
    """Negative-minus-positive gap of mean Skew, with a two-sided permutation p-value over words."""
    s = d["skew"].to_numpy(); neg = (d["category"] == "negative").to_numpy()
    obs = s[neg].mean() - s[~neg].mean()
    rng = np.random.default_rng(seed)
    null = np.empty(n_perm)
    for i in range(n_perm):
        p = rng.permutation(neg)
        null[i] = s[p].mean() - s[~p].mean()
    return obs, float((np.abs(null) >= abs(obs)).mean())

rows = []
pn = r[r["category"] != "socioeconomic"]
for g in RACES:
    dg = pn[pn["group"] == g]
    for t in TEMPLATES + ["mean over templates"]:
        d = dg[dg["template"] == t] if t in TEMPLATES else \
            dg.groupby(["concept", "category"], as_index=False)["skew"].mean()
        gap, p = gap_test(d)
        rows.append(dict(group=g, template=t, level=d["skew"].mean(), gap=gap, p=p))
gaps = pd.DataFrame(rows)
gaps["p_adj"] = benjamini_hochberg(gaps["p"].to_numpy())
gaps["sig"] = gaps["p_adj"] < 0.05
tables["A3_level_and_gap"] = gaps

print("LEVEL: mean Skew over all words")
display(gaps.pivot(index="template", columns="group", values="level").loc[TEMPLATES + ["mean over templates"], RACES])
print("GAP: negative minus positive (* = significant after correction)")
display(gaps.assign(cell=gaps["gap"].round(2).astype(str) + np.where(gaps["sig"], " *", ""))
            .pivot(index="template", columns="group", values="cell").loc[TEMPLATES + ["mean over templates"], RACES])

**How to read it.** If a group's gap has the **same sign under every template**, the positive/negative
difference is not caused by one template, even if that template makes it larger. If the sign flips
between templates, nothing should be claimed at the word level for that group. The level table and the
gap table can disagree: a template can push a group out for *all* words (level) without creating the
good/bad difference (gap).

## 4. Level 4 → 5: from groups to one KPI

The KPIs reported so far are **mean MaxSkew**, **mean MinSkew** and **mean NDKL** over prompts. Three checks:

1. **Leave one template out.** Does the KPI change if a template is removed? If not, the KPI cannot see
   template effects at all.
2. **Who drives the KPI?** MaxSkew takes, for each prompt, the most over-represented group; MinSkew the most
   under-represented one. Counting which group that is shows whose bias the KPI is really measuring.
3. **Order of aggregation.** "Mean over prompts of the per-prompt maximum" is what we report. "Maximum over
   groups of the per-group mean" is another, equally natural, way to summarise the same numbers.

In [ ]:
q = per_query.query("query_set == @QS and reference == @REF and k == @K")
kpi_cols = ["max_skew", "min_skew", "ndkl"]

# 1. leave one template out
loo = {"(all templates)": q.groupby("attribute")[kpi_cols].mean()}
for t in TEMPLATES:
    loo[f"without '{t}'"] = q[q["template"] != t].groupby("attribute")[kpi_cols].mean()
loo = pd.concat(loo, names=["dropped"]).unstack("attribute")
tables["A4_kpi_leave_one_template_out"] = loo
display(loo)

# 2. which group is the max / min for each prompt (race)
drivers = pd.DataFrame({
    "MaxSkew group (most over-represented)": r.loc[r.groupby("text")["skew"].idxmax(), "group"].value_counts(),
    "MinSkew group (most under-represented)": r.loc[r.groupby("text")["skew"].idxmin(), "group"].value_counts(),
}).reindex(RACES).fillna(0).astype(int)
tables["A4_kpi_drivers_race"] = drivers
display(drivers)

# 3. order of aggregation (race)
by_group = r.groupby("group")["skew"].mean()
print(f"mean over prompts of max Skew : {r.groupby('text')['skew'].max().mean():.2f}   (what we report as MaxSkew)")
print(f"max over groups of mean Skew  : {by_group.max():.2f}   ({by_group.idxmax()})")
print(f"mean over prompts of min Skew : {r.groupby('text')['skew'].min().mean():.2f}   (what we report as MinSkew)")
print(f"min over groups of mean Skew  : {by_group.min():.2f}   ({by_group.idxmin()})")

### Result: Part A (computed from notebook 03's saved results)

*These numbers come from `baseline_v1_berg_results.zip`, so they will be identical when you rerun the cells.*

**Level 0–2 (single prompts and template average).** "A friendly person" and "A unfriendly person" give White
exactly the same Skew, −3.63, the floor value for zero White faces in the top 100. Middle Eastern is also at
its floor (−3.08) for both. With "A photo of a {} person", White is still low (−1.15 for both words) but not
absent. The template average in the notebook 03 heatmap hides all of this.

**Level 3 (positive vs negative).** This corrects what we concluded on 19 November.
- "A {} person" changes White's **level**: −1.27 there, between −0.61 and +0.56 under the other templates.
- It does **not** create White's positive/negative **gap**. White is higher on negative than on positive
  words under **all 8 templates** (from +0.17 to +1.44), significant after correction under 5 of them and
  on the template average (+0.67). "A {} person" has the largest gap, but it is +0.86 with
  "A photo of a {} person" and +0.96 with "This is a photo of a {} person" too.
- The same holds for Middle Eastern (higher on negative words under all 8 templates, +0.59 on average).
  Indian (−0.65) and Southeast Asian (−0.48) show the opposite: lower on negative words under all 8 templates.
  Black, East Asian and Latino_Hispanic show no consistent gap.

*So "the White column depends on the phrasing, so it shows no reliable word-level pattern" was only half right.
The **level** depends on the phrasing; the **direction of the gap** does not. A safer sentence for the report:
"White and Middle Eastern faces are retrieved more for negative than for positive adjectives, and Indian and
Southeast Asian faces less, under every template; the size of the gap, and White's overall level, depend
strongly on the template." The test treats the 31 words as exchangeable, which they are not quite, so these
are indications to confirm after mitigation, not final claims.*

**Level 4–5 (KPI).**
- Mean race MaxSkew@100 is 0.76 and stays between 0.75 and 0.78 whichever template is removed. The KPI is
  blind to the template effect that moves White's level from −1.27 to +0.56.
- MaxSkew is driven by East Asian (91 of 264 prompts), White (75) and Southeast Asian (64). MinSkew is
  driven almost only by Indian (94) and Black (90). The two KPIs measure different groups' bias.
- The order of aggregation matters a lot: the mean of per-prompt maxima is 0.76, the maximum of per-group
  means is 0.29 (East Asian). Taking a maximum per prompt picks up noise and template effects, which then
  add up instead of cancelling out.

*This means a single aggregated KPI is fine for comparing methods (it is stable), but it cannot explain
results. For the thesis we should report the KPI together with (i) the per-group level, (ii) which groups
drive MaxSkew/MinSkew, and (iii) the per-template spread.*

# Part B · Why does "A {} person" push White faces out? (GPU)

Part A shows **that** the template moves groups. Part B looks for **why**:

| Section | Check | If yes, it means |
|---|---|---|
| 6 | Look at single prompts (top faces + Skew per group) | what the professors asked to see |
| 7 | Does the adjective matter? Do capital letters? | the effect is in the sentence frame itself |
| 8 | Is the whole White score distribution shifted, or only the top? | a global shift vs a ranking detail |
| 9 | Does the "template direction" in embedding space line up with groups? | a geometric explanation of the effect |
| 10 | Is that direction about race or about image style (brightness, colour, contrast)? | a dataset confound rather than race |

Run on a T4 GPU. Encoding text is cheap; only the gallery needs encoding once (cached).

In [ ]:
import torch
from vlbias.utils.config import load_yaml, set_seed
from vlbias.models.load import load_model
from vlbias.data.fairface import load_gallery
from vlbias.data.attributes import resolve_attributes
from vlbias.retrieval.embed import gallery_embeddings, encode_texts
from vlbias.retrieval.rank import top_k_indices
from vlbias.metrics.fairness import skew_at_k

CONFIG = "configs/experiments/pipeline_v1.yaml"     # notebook 03's setup (Berg et al. protocol)
USE_DRIVE = False

cfg = load_yaml(CONFIG)
CACHE_DIR = Path(cfg["cache_dir"])
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/thesis/embeddings")
set_seed(cfg["seed"])
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch to a T4!")

vlm = load_model(load_yaml(cfg["model"]))
gallery = load_gallery(cfg["dataset"], cfg["seed"])
labels, names = resolve_attributes(gallery, cfg["attributes"])
g_emb = gallery_embeddings(vlm, gallery, CACHE_DIR, batch_size=cfg["batch_size"])
enc = lambda texts: encode_texts(vlm, list(texts))
print(gallery.name, g_emb.shape)

## 5. Live Skew for any prompt, and a check against notebook 03

`skews(prompts)` computes Skew@100 per group for any list of prompts, with exactly the definition used in
notebook 03 (gallery reference, zero floor 0.5; for `age_adult` only adults are searched). The check cell
recomputes three saved prompts: it must print **OK**, otherwise the gallery or the embeddings differ from
notebook 03 and nothing below can be compared with it.

In [ ]:
ZERO_FLOOR = cfg["metrics"]["zero_floor"]

def skews(texts, attribute="race", k=K, reference=REF):
    """Skew@k per group (columns) for each prompt (rows), same definition as notebook 03."""
    lab = labels[attribute]; ok = lab >= 0
    L, G = lab[ok], g_emb[ok]
    n = len(names[attribute])
    desired = np.bincount(L, minlength=n) / len(L) if reference == "gallery" else np.full(n, 1 / n)
    top = top_k_indices(enc(texts), G, k)
    return pd.DataFrame([skew_at_k(L[t], desired, k, ZERO_FLOOR) for t in top],
                        index=list(texts), columns=names[attribute])

check = ["A friendly person", "A photo of a kind person", "This person is attractive"]
live = skews(check)
saved = pd.concat({t: one_query(t)["skew"] for t in check}, axis=1).T[live.columns]
diff = float((live - saved).abs().max().max())
print(f"largest difference: {diff:.2e}")
print("OK: live == notebook 03" if diff < 1e-3 else "MISMATCH: different gallery or embeddings, stop here")

## 6. Single-prompt viewer

The professors asked to **see** single-prompt results. `show(prompts)` puts the top images of several
prompts in rows, with race / gender / age under each face, and prints the Skew of every group.

In [ ]:
SHORT = {"Latino_Hispanic": "Latino", "Southeast Asian": "SE Asian", "Middle Eastern": "M. Eastern",
         "East Asian": "E. Asian", "more than 70": "70+"}

def label_of(i):
    return " | ".join(SHORT.get(n, n) for n in
                      (gallery.label_names[a][gallery.labels[a][i]] for a in ("race", "gender", "age")))

def show_idx(rows, k=8, size=1.7):
    """rows: {title: list of gallery indices}"""
    fig, axes = plt.subplots(len(rows), k, figsize=(size * k, size * 1.25 * len(rows)), squeeze=False)
    for ax_row, (title, idx) in zip(axes, rows.items()):
        for ax, i in zip(ax_row, list(idx)[:k]):
            ax.imshow(gallery.images[int(i)]["image"]); ax.set_xticks([]); ax.set_yticks([])
            ax.set_xlabel(label_of(int(i)), fontsize=6)
        ax_row[0].set_ylabel(title, fontsize=8, rotation=0, ha="right", va="center")
    plt.tight_layout(); plt.show()

def show(prompts, k=8, attribute="race"):
    top = top_k_indices(enc(prompts), g_emb, k)
    show_idx({p: t for p, t in zip(prompts, top)}, k)
    display(skews(prompts, attribute).round(2))

show(["A kind person", "A photo of a kind person", "A unkind person", "A photo of a unkind person"])

## 7. Does the adjective matter? Frame-only prompts

If "A person", with no adjective at all, already pushes White faces out, the effect lives in the sentence
frame and the words do not matter. The capitalisation check confirms that "A" and "a" are the same input:
CLIP's tokenizer lowercases everything, so capital letters cannot be the cause.

In [ ]:
a, b = "A kind person", "a kind person"
same_tokens = np.array_equal(np.asarray(vlm.tokenizer([a])), np.asarray(vlm.tokenizer([b])))
print(f"same tokens: {same_tokens} | cosine of the two text embeddings: {float(enc([a])[0] @ enc([b])[0]):.4f}")

ladder = {
    "frame only":   ["A person", "An individual", "A photo of a person", "This is a photo of a person",
                     "This is the face of a person", "A cropped photo of a face", "A face", "A photo"],
    "other nouns":  ["A man", "A woman", "A human", "A human being", "A photo of a man", "A photo of a woman"],
    "frame + word": ["A kind person", "A photo of a kind person", "A unkind person", "A photo of a unkind person"],
}
lad = pd.concat({kind: skews(ps) for kind, ps in ladder.items()}, names=["type", "prompt"])
tables["B7_frame_ladder"] = lad
display(lad.style.format("{:.2f}").background_gradient(cmap="RdBu_r", vmin=-2, vmax=2, axis=None))

**How to read it.** Look at the White column. If "A person" and "An individual" are as low as "A kind person",
while "A photo of a person" is not, the cause is the **short frame without "photo"/"face"**, independent of the
adjective. "A man" / "A woman" show whether the noun "person" matters.

## 8. Score distributions: shifted everywhere, or only at the top?

Skew@100 only looks at the top 100 of ~10,000 faces. Here we look at the similarity of **every** face to the
prompt, per group. The dashed line is the score of the 100th face: only faces above it count in Skew@100.

In [ ]:
def score_table(prompts, attribute="race", k=K):
    lab = labels[attribute]; ok = lab >= 0
    S = enc(prompts) @ g_emb[ok].T
    rows = []
    for p, s in zip(prompts, S):
        thr = np.sort(s)[-k]
        z = (s - s.mean()) / s.std()
        for gi, g in enumerate(names[attribute]):
            m = lab[ok] == gi
            rows.append(dict(prompt=p, group=g, mean_z=z[m].mean(), share_above_top_k=(s[m] >= thr).mean() * 100))
    return S, pd.DataFrame(rows)

def plot_scores(prompts, attribute="race", k=K):
    lab = labels[attribute]; ok = lab >= 0
    S, _ = score_table(prompts, attribute, k)
    fig, axes = plt.subplots(1, len(prompts), figsize=(4 * len(prompts), 3.2), sharey=False)
    for ax, p, s in zip(np.atleast_1d(axes), prompts, S):
        ax.boxplot([s[lab[ok] == gi] for gi in range(len(names[attribute]))], showfliers=False)
        ax.axhline(np.sort(s)[-k], ls="--", color="C3", lw=1)
        ax.set_xticks(range(1, len(names[attribute]) + 1),
                      [SHORT.get(n, n) for n in names[attribute]], rotation=60, ha="right", fontsize=8)
        ax.set_title(f'"{p}"', fontsize=9)
    np.atleast_1d(axes)[0].set_ylabel("cosine similarity")
    plt.tight_layout(); plt.show()

dist_prompts = ["A kind person", "A photo of a kind person", "A person", "A photo of a person"]
plot_scores(dist_prompts)
_, st = score_table(dist_prompts)
tables["B8_score_distributions"] = st
print("mean similarity per group, in std units of that prompt (0 = average face)")
display(st.pivot(index="prompt", columns="group", values="mean_z").loc[dist_prompts, names["race"]])
print("% of each group's faces above the top-100 threshold")
display(st.pivot(index="prompt", columns="group", values="share_above_top_k").loc[dist_prompts, names["race"]].round(2))

**How to read it.** If the whole White box moves down under "A kind person" (mean_z clearly negative), CLIP
scores White faces lower overall for that frame: a **global shift**. If the boxes look the same and only the
top differs, the effect is a **tail effect** on a few images, which is much more fragile.

## 9. Geometry: the "template direction"

For each template *t*, average its text embedding over the 33 words: **E_t**. The difference between E_t and
the average over all templates, **d_t = E_t − mean_t E_t**, is the "template direction". The score of a face
*x* for a prompt is *x · e*, so **x · d_t** is exactly how much template *t* raises or lowers face *x*
compared with the other templates, whatever the word. If White faces have a low average x · d_t for
"A {} person", that is the mechanism behind the notebook 03 result.

The cell also compares this geometric prediction with the observed per-template level from Part A.

In [ ]:
E = {t: enc([t.format(w) for w in WORDS]).mean(0) for t in TEMPLATES}
center = np.mean(list(E.values()), axis=0)
proj = {t: g_emb @ (E[t] - center) for t in TEMPLATES}

race_lab = labels["race"]
def group_means(v, lab=race_lab, groups=names["race"]):
    z = (v - v.mean()) / v.std()
    return pd.Series([z[lab == gi].mean() for gi in range(len(groups))], index=groups)

geo = pd.DataFrame({t: group_means(proj[t]) for t in TEMPLATES}).T[RACES]
level = gaps.pivot(index="template", columns="group", values="level").loc[TEMPLATES, RACES]
tables["B9_template_direction_by_group"] = geo
print("mean projection on the template direction (std units)")
display(geo.style.format("{:.2f}").background_gradient(cmap="RdBu_r", vmin=-0.6, vmax=0.6, axis=None))

lvl_c = level - level.mean()          # remove each group's overall level, keep the template part
fig, ax = plt.subplots(figsize=(4.5, 4))
for g in RACES:
    ax.scatter(geo[g], lvl_c[g], label=g, s=25)
ax.axhline(0, color="0.7", lw=0.6); ax.axvline(0, color="0.7", lw=0.6)
ax.set_xlabel("geometry: mean projection on d_t"); ax.set_ylabel("observed: level − group mean (Part A)")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()
print(f"correlation geometry vs observed: {np.corrcoef(geo.values.ravel(), lvl_c.values.ravel())[0, 1]:.2f}")

d_word = E["A {} person"] - E["A photo of a {} person"]
d_frame = enc(["A person"])[0] - enc(["A photo of a person"])[0]
print(f"cosine('A {{}} person' - 'A photo of a {{}} person' over words, 'A person' - 'A photo of a person'): "
      f"{d_word @ d_frame / np.linalg.norm(d_word) / np.linalg.norm(d_frame):.2f}")

**How to read it.** The scatter compares the geometric prediction (x axis) with what notebook 03 measured
(y axis), one dot per template × group. A high correlation means the template effect is fully explained by
one direction in embedding space. The last line says whether this direction is the same with and without
the adjective (close to 1 = the words play no role).

## 10. Race or image style? Low-level image statistics

FairFace images come from different sources, so groups can differ in brightness, colour or contrast. If the
template direction mostly tracks these, the "White" effect is really a **photo-style** effect that happens
to correlate with race in this dataset.

We compute four simple statistics per image (on a 64×64 copy, cached to CSV) and ask two questions:

1. Within each race group, does the projection on the "A {} person" direction correlate with them?
2. Does the White gap in the projection shrink once these statistics are controlled for (linear regression)?

These are crude stand-ins for image style (they say nothing about skin tone, which FairFace does not label).

In [ ]:
def image_stats(img):
    im = img.convert("RGB").resize((64, 64))
    rgb = np.asarray(im, dtype=np.float32) / 255
    sat = np.asarray(im.convert("HSV"), dtype=np.float32)[..., 1] / 255
    gray = rgb.mean(2)
    return dict(brightness=gray.mean(), contrast=gray.std(), saturation=sat.mean(),
                center_brightness=gray[16:48, 16:48].mean())

stats_path = OUT / f"image_stats_{gallery.name}.csv"
if stats_path.exists():
    stats = pd.read_csv(stats_path)
else:
    stats = pd.DataFrame([image_stats(gallery.images[i]["image"]) for i in range(len(gallery))])
    stats.to_csv(stats_path, index=False)
STAT_COLS = list(stats.columns)

print("mean image statistics per race group")
display(stats.groupby(np.array(names["race"])[race_lab]).mean().loc[RACES])

In [ ]:
def ols(y, X):
    X = np.column_stack([np.ones(len(y)), X])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    r2 = 1 - ((y - X @ beta) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return beta, r2

def confound_check(v, group, lab=race_lab, groups=names["race"]):
    """Is the group's gap on projection v explained by image statistics?"""
    y = (v - v.mean()) / v.std()
    is_g = (lab == groups.index(group)).astype(float)
    S = ((stats - stats.mean()) / stats.std()).to_numpy()
    b1, r2_1 = ols(y, is_g[:, None])
    b2, r2_2 = ols(y, np.column_stack([is_g, S]))
    within = pd.DataFrame({g: [np.corrcoef(y[lab == gi], stats[c][lab == gi])[0, 1] for c in STAT_COLS]
                           for gi, g in enumerate(groups)}, index=STAT_COLS)
    summary = pd.Series({f"{group} gap, group only": b1[1], f"{group} gap, + image stats": b2[1],
                         "R2, group only": r2_1, "R2, group + image stats": r2_2})
    return summary, within

summary, within = confound_check(proj["A {} person"], "White")
tables["B10_confound_summary"] = summary; tables["B10_within_group_corr"] = within
display(summary.round(3).to_frame("value"))
print("within-group correlation between the projection and each image statistic")
display(within[RACES].round(2))

order = np.argsort(proj["A {} person"])
show_idx({"lowest on\n'A {} person'\ndirection": order[:8], "highest": order[::-1][:8]})

**How to read it.** If the White gap is about the same with and without image statistics, and the
within-group correlations are small, the direction is about something CLIP associates with race (or a
feature we did not measure). If the gap shrinks a lot and the correlations are clear (e.g. brightness), part
of the effect is photo style. The two rows of faces show what the direction "likes" and "dislikes": look
for a pattern in lighting, colour, crop or background, not only in race.

# Part C · Indian faces and appearance words (GPU)

In notebook 03, attractive (−1.83) and unattractive (−1.79) both pushed Indian faces **below their usual
level**, consistently across templates. Three possible explanations, each with a check:

| Section | Explanation | Check |
|---|---|---|
| 11 | CLIP barely separates a word from its negation | text similarity and top-100 overlap of opposite pairs |
| 12 | It is the **topic** (looks), not the words with "un-" | other appearance words, with and without negation |
| 13 | Appearance words point to features Indian faces in FairFace have less of | an "appearance direction", as in section 9 |

## 11. Opposite pairs: does CLIP tell them apart?

For each pair, averaged over the 8 templates:
- **text cosine**: similarity of the two prompts' text embeddings;
- **top-100 overlap**: share of faces that both prompts retrieve (Jaccard);
- **move together**: correlation, across the 7 race groups, of the two words' effects in notebook 03
  (Skew minus the group's usual level). +1 = same faces in and out, −1 = opposite.

The baseline row is the average over all *unrelated* word pairs, so the numbers can be read against it.

In [ ]:
PAIRS = {
    "prefix (un-/non-/im-/dis-)": [("attractive", "unattractive"), ("friendly", "unfriendly"),
                                    ("successful", "unsuccessful"), ("kind", "unkind"),
                                    ("noncriminal", "criminal"), ("nonviolent", "violent"),
                                    ("moral", "immoral"), ("honest", "dishonest")],
    "different word": [("smart", "dumb"), ("clever", "stupid"), ("hardworking", "lazy"), ("good", "evil"),
                       ("trustworthy", "dangerous"), ("heroic", "villainous"), ("rich", "poor")],
}
W_E = {w: enc([t.format(w) for t in TEMPLATES]) for w in WORDS}               # 8 x d per word
W_TOP = {w: top_k_indices(W_E[w], g_emb, K) for w in WORDS}                    # 8 x K per word
m03 = r.groupby(["concept", "group"])["skew"].mean().unstack()[RACES]
effect = m03 - m03.mean()                                                      # minus the group's usual level

def pair_row(a, b):
    cos = float((W_E[a] * W_E[b]).sum(1).mean())
    jac = float(np.mean([len(set(x) & set(y)) / len(set(x) | set(y)) for x, y in zip(W_TOP[a], W_TOP[b])]))
    return dict(text_cos=cos, top100_overlap=jac, move_together=np.corrcoef(effect.loc[a], effect.loc[b])[0, 1],
                indian_a=effect.loc[a, "Indian"], indian_b=effect.loc[b, "Indian"])

rows = [dict(type=kind, pair=f"{a} / {b}", **pair_row(a, b)) for kind, ps in PAIRS.items() for a, b in ps]
listed = {frozenset(p) for ps in PAIRS.values() for p in ps}
unrel = [pair_row(a, b) for i, a in enumerate(WORDS) for b in WORDS[i + 1:] if frozenset((a, b)) not in listed]
rows.append(dict(type="baseline", pair="unrelated word pairs (mean)", **pd.DataFrame(unrel).mean().to_dict()))
pairs = pd.DataFrame(rows).set_index(["type", "pair"])
tables["C11_opposite_pairs"] = pairs
display(pairs.style.format("{:.2f}").background_gradient(subset=["text_cos", "top100_overlap", "move_together"],
                                                         cmap="Reds"))

**How to read it.** If prefix pairs have a text cosine and overlap clearly **above** the unrelated baseline and
above the "different word" pairs, CLIP treats "un-X" as almost "X" (the negation is mostly lost). If some prefix
pairs (e.g. kind/unkind) move in opposite directions, the negation is **not** always lost, and "CLIP cannot
handle negation" is too strong a claim: it depends on the word. `indian_a` and `indian_b` are the two words'
effects on Indian faces.

## 12. Is it the topic? Appearance words with and without negation

If Indian faces drop for **every** appearance word (beautiful, ugly, good-looking, plain-looking, "not
attractive", ...) and not for the control words, the cause is the **topic of looks**, not negation. Each word
is put into the 8 templates; the table shows the Indian word effect (Skew minus Indian's usual level under
the same template, averaged over templates) and its std over templates (reliable if |mean| is clearly larger
than the std, as in notebook 03).

In [ ]:
APPEARANCE = ["attractive", "unattractive", "not attractive", "beautiful", "ugly",
              "good-looking", "plain-looking", "pretty", "handsome", "photogenic"]
CONTROL = ["kind", "honest", "friendly", "tall", "young", "old"]

usual = (skews([t.format(w) for t in TEMPLATES for w in WORDS])
         .assign(template=[t for t in TEMPLATES for w in WORDS]).groupby("template").mean())

def word_effects(words):
    """Race word effect per word: mean and std over the 8 templates of (Skew - usual level)."""
    mean, std = {}, {}
    for w in words:
        s = skews([t.format(w) for t in TEMPLATES])
        s.index = TEMPLATES
        e = s - usual.loc[TEMPLATES, s.columns]
        mean[w], std[w] = e.mean(), e.std()
    return pd.DataFrame(mean).T[RACES], pd.DataFrame(std).T[RACES]

WTYPE = pd.Series(["appearance"] * len(APPEARANCE) + ["control"] * len(CONTROL), index=APPEARANCE + CONTROL)
we_mean, we_std = word_effects(APPEARANCE + CONTROL)
indian = pd.DataFrame({"type": WTYPE, "effect": we_mean["Indian"], "std": we_std["Indian"]})
indian["reliable"] = indian["effect"].abs() > indian["std"]
tables["C12_word_effects_race_mean"] = we_mean.assign(type=WTYPE)
tables["C12_word_effects_race_std"] = we_std.assign(type=WTYPE)
print("Indian: word effect (mean over templates), its std, and whether |mean| > std")
display(indian.round(2))
print("all groups: mean word effect")
display(we_mean.set_index(WTYPE, append=True)
          .style.format("{:.2f}").background_gradient(cmap="RdBu_r", vmin=-2, vmax=2, axis=None))

**How to read it.** In the second table, look at where the Indian share goes: the groups that go **up** for
appearance words are the ones CLIP associates with the topic of looks. "young" and "old" are controls that
are about looks in a broad sense but not about beauty; if they behave like the appearance words, the topic
is "physical description" rather than "beauty". Below, the same for gender and age: appearance words often
pull in young women, which can push other groups out indirectly.

In [ ]:
probe = ["attractive", "unattractive", "beautiful", "ugly", "kind", "honest"]
for attr in ("gender", "age_adult"):
    t = pd.DataFrame({w: skews([tt.format(w) for tt in TEMPLATES], attr).mean() for w in probe}).T
    tables[f"C12_appearance_{attr}"] = t
    print(attr, "- mean Skew@100 over the 8 templates"); display(t.round(2))

## 13. The "appearance direction"

As in section 9, but for the topic: **a** = mean embedding of the appearance prompts − mean embedding of the
control prompts (same 8 templates). x · a says how much a face gains when a prompt is about looks rather than
about character. If Indian faces score low on **a**, appearance words of either polarity push them out. The
same confound check as section 10 tells whether **a** follows image style.

In [ ]:
A_emb = enc([t.format(w) for t in TEMPLATES for w in APPEARANCE]).mean(0)
C_emb = enc([t.format(w) for t in TEMPLATES for w in ["kind", "honest", "friendly"]]).mean(0)
appear = g_emb @ (A_emb - C_emb)

ap = group_means(appear)[RACES]
tables["C13_appearance_direction_by_group"] = ap
print("mean projection on the appearance direction (std units)"); display(ap.round(2).to_frame("mean_z").T)

summary, within = confound_check(appear, "Indian")
tables["C13_confound_summary"] = summary; tables["C13_within_group_corr"] = within
display(summary.round(3).to_frame("value"))
display(within[RACES].round(2))

order = np.argsort(appear)
show_idx({"lowest on\nappearance\ndirection": order[:8], "highest": order[::-1][:8]})

**How to read it.** A clearly negative Indian value means the topic of looks, as CLIP encodes it, points away
from Indian faces in FairFace. Then "attractive" and "unattractive" both exclude Indian faces because they
share this direction, and the "un-" barely changes it (section 11). The confound check and the two rows of
faces show whether this is about the faces themselves or about how the photos look.

## 14. Save

All tables of this notebook as CSV, plus the image statistics, zipped for download. Keep the zip outside the
repo, as with notebook 03.

In [ ]:
for name, t in tables.items():
    t.to_csv(OUT / f"{name}.csv")
print(len(tables), "tables saved in", OUT)
!cd {OUT.parent} && zip -q -r /content/notebook05_results.zip {OUT.name}
from google.colab import files
files.download("/content/notebook05_results.zip")

## 15. Summary (fill in after running Parts B and C)

**Aggregation (Part A).** *(results above)* The template changes a group's level; the KPI does not see it;
MaxSkew and MinSkew are driven by different groups; the order of aggregation changes the number by more than
2×. White's positive/negative gap has the same sign under all templates.

**"A {} person" (Part B).**
- Does the effect need an adjective? (section 7) …
- Global shift or top only? (section 8) …
- Is it explained by the template direction? correlation = … (section 9)
- Race or image style? White gap … → … with image stats (section 10)

**Indian and appearance words (Part C).**
- Do prefix pairs collapse more than different-word pairs? (section 11) …
- Do all appearance words exclude Indian faces, including "beautiful"/"ugly"? (section 12) …
- Indian mean on the appearance direction = …; explained by image stats? (section 13) …